In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import LeaveOneOut
from joblib import Parallel, delayed
import xgboost as xgb
import multiprocessing
import warnings
warnings.filterwarnings("ignore")

# ======================================================
# 1) Load data
# ======================================================
file_path = "C:/Users/aiden/School/6380/in-silico-drug-discovery/data/input_data.csv"
df = pd.read_csv(file_path)
labeled = df[df["senolytic"].isin([0, 1])].copy()
X = labeled.drop(columns=["senolytic", "ID", "SMILES"])
y = labeled["senolytic"].values

# ======================================================
# 2) Scale features
# ======================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
n_jobs = multiprocessing.cpu_count()
loo = LeaveOneOut()

# ======================================================
# 3) LOOCV function
# ======================================================
def run_loocv(X_sel, y, pos_weight=1, max_depth=3, verbose_prefix=""):
    """Run LOOCV for XGBoost with sample weights and max_depth"""
    def run_fold(train_idx, test_idx):
        X_train, X_test = X_sel[train_idx], X_sel[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        sample_weights = np.where(y_train == 1, pos_weight, 1)

        model = xgb.XGBClassifier(
            use_label_encoder=False,
            eval_metric="logloss",
            objective="binary:logistic",
            max_depth=max_depth,
            n_estimators=200,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=0,
            n_jobs=1
        )
        model.fit(X_train, y_train, sample_weight=sample_weights)
        y_hat = model.predict(X_test)
        return y_test[0], y_hat[0]

    results = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(run_fold)(train_idx, test_idx)
        for train_idx, test_idx in loo.split(X_sel)
    )

    y_true, y_pred = zip(*results)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"{verbose_prefix}LOOCV completed | F1={f1:.4f}")
    return precision, recall, f1

# ======================================================
# 4) Step 1: Sweep positive class weights (all features, max_depth=3)
# ======================================================
weight_values = np.linspace(1, 500, 20, dtype=int)
weight_results = []
print("Starting positive class weight sweep (all features, max_depth=3)...")
for w in weight_values:
    print(f"\nEvaluating weight={w}...")
    precision, recall, f1 = run_loocv(X_scaled, y, pos_weight=w, max_depth=3, verbose_prefix=f"[Weight={w}] ")
    weight_results.append((w, precision, recall, f1))

best_w_idx = np.argmax([r[3] for r in weight_results])
best_weight = weight_results[best_w_idx][0]
print(f"\nBest positive class weight: {best_weight} | F1={weight_results[best_w_idx][3]:.4f}\n")

# ======================================================
# 5) Step 2: Feature selection with best weight
# ======================================================
print("Computing feature importances via Random Forest...")
rf = RandomForestClassifier(n_estimators=500, n_jobs=-1, class_weight="balanced")
rf.fit(X_scaled, y)
importances = rf.feature_importances_
feat_idx_sorted = np.argsort(importances)[::-1]
print("Feature importances computed.\n")

top_k_values = list(range(20, 201, 20))
f1_topk_results = []

print("Starting top_k feature sweep with best weight...")
for top_k in top_k_values:
    print(f"\nEvaluating top_k={top_k} features...")
    top_features_idx = feat_idx_sorted[:top_k]
    X_selected = X_scaled[:, top_features_idx]
    _, _, f1 = run_loocv(X_selected, y, pos_weight=best_weight, max_depth=3, verbose_prefix=f"[Top_k={top_k}] ")
    f1_topk_results.append(f1)

best_topk = top_k_values[np.argmax(f1_topk_results)]
X_selected = X_scaled[:, feat_idx_sorted[:best_topk]]
print(f"\nBest top_k features: {best_topk} | F1={f1_topk_results[np.argmax(f1_topk_results)]:.4f}\n")

# ======================================================
# 6) Step 3: Max depth sweep with best weight & features
# ======================================================
depth_values = range(1, 11)
f1_depth_results = []

print("Starting max_depth sweep...")
for depth in depth_values:
    print(f"\nEvaluating max_depth={depth}...")
    _, _, f1 = run_loocv(X_selected, y, pos_weight=best_weight, max_depth=depth, verbose_prefix=f"[Depth={depth}] ")
    f1_depth_results.append(f1)

best_depth = depth_values[np.argmax(f1_depth_results)]
print(f"\nBest max_depth: {best_depth} | F1={f1_depth_results[np.argmax(f1_depth_results)]:.4f}\n")

# ======================================================
# 7) Final model training on best weight, features, depth
# ======================================================
best_model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss",
    objective="binary:logistic",
    max_depth=best_depth,
    n_estimators=200,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
best_model.fit(X_selected, y)

print("Final model trained on best weight, top features, and max depth.")

Starting positive class weight sweep (all features, max_depth=3)...

Evaluating weight=1...
[Weight=1] LOOCV completed | F1=0.2119

Evaluating weight=27...
[Weight=27] LOOCV completed | F1=0.2805

Evaluating weight=53...
[Weight=53] LOOCV completed | F1=0.2687

Evaluating weight=79...
[Weight=79] LOOCV completed | F1=0.2721

Evaluating weight=106...
[Weight=106] LOOCV completed | F1=0.2658

Evaluating weight=132...
[Weight=132] LOOCV completed | F1=0.2733

Evaluating weight=158...
[Weight=158] LOOCV completed | F1=0.2746

Evaluating weight=184...
[Weight=184] LOOCV completed | F1=0.2722

Evaluating weight=211...
[Weight=211] LOOCV completed | F1=0.2678

Evaluating weight=237...
[Weight=237] LOOCV completed | F1=0.2582

Evaluating weight=263...
[Weight=263] LOOCV completed | F1=0.2527

Evaluating weight=289...
[Weight=289] LOOCV completed | F1=0.2730

Evaluating weight=316...
[Weight=316] LOOCV completed | F1=0.2553

Evaluating weight=342...
[Weight=342] LOOCV completed | F1=0.2467

Eva

In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from sklearn.model_selection import LeaveOneOut
from joblib import Parallel, delayed
import xgboost as xgb
import multiprocessing
import warnings
warnings.filterwarnings("ignore")

# ======================================================
# Load data
# ======================================================
file_path = "C:/Users/aiden/School/6380/in-silico-drug-discovery/data/input_data.csv"
df = pd.read_csv(file_path)

labeled = df[df["senolytic"].isin([0, 1])].copy()
X = labeled.drop(columns=["senolytic", "ID", "SMILES"])
y = labeled["senolytic"].values

# ======================================================
# Scale
# ======================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ======================================================
# Feature selection (top 160)
# ======================================================
rf = RandomForestClassifier(n_estimators=500, n_jobs=-1, class_weight="balanced")
rf.fit(X_scaled, y)

importances = rf.feature_importances_
feat_idx_sorted = np.argsort(importances)[::-1]

top_k = 160
X_selected = X_scaled[:, feat_idx_sorted[:top_k]]

# ======================================================
# LOOCV setup
# ======================================================
loo = LeaveOneOut()
n_jobs = multiprocessing.cpu_count()

# ======================================================
# LOOCV evaluation
# ======================================================
def run_fold(train_idx, test_idx):
    X_train, X_test = X_selected[train_idx], X_selected[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    sample_weights = np.where(y_train == 1, 27, 1)

    model = xgb.XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        objective="binary:logistic",
        max_depth=3,
        n_estimators=200,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0,
        n_jobs=1
    )

    model.fit(X_train, y_train, sample_weight=sample_weights)
    
    y_pred = model.predict(X_test)[0]
    y_prob = model.predict_proba(X_test)[0, 1]

    return y_test[0], y_pred, y_prob

print("Running final LOOCV evaluation...")

results = Parallel(n_jobs=n_jobs, verbose=10)(
    delayed(run_fold)(train_idx, test_idx)
    for train_idx, test_idx in loo.split(X_selected)
)

# ======================================================
# Collect results
# ======================================================
y_true, y_pred, y_prob = zip(*results)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

# ======================================================
# Metrics
# ======================================================
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

print("\n===== FINAL LOOCV PERFORMANCE =====")
print(f"AUC      : {auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"Accuracy : {accuracy:.4f}")

Running final LOOCV evaluation...


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   9 tasks      | elapsed:    1.8s
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    3.3s
[Parallel(n_jobs=16)]: Done  29 tasks      | elapsed:    3.6s
[Parallel(n_jobs=16)]: Done  40 tasks      | elapsed:    5.0s
[Parallel(n_jobs=16)]: Done  53 tasks      | elapsed:    6.5s
[Parallel(n_jobs=16)]: Done  66 tasks      | elapsed:    8.0s
[Parallel(n_jobs=16)]: Done  81 tasks      | elapsed:    9.3s
[Parallel(n_jobs=16)]: Done  96 tasks      | elapsed:   10.3s
[Parallel(n_jobs=16)]: Done 113 tasks      | elapsed:   12.2s
[Parallel(n_jobs=16)]: Done 130 tasks      | elapsed:   13.9s
[Parallel(n_jobs=16)]: Done 149 tasks      | elapsed:   15.9s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:   17.6s
[Parallel(n_jobs=16)]: Done 189 tasks      | elapsed:   19.4s
[Parallel(n_jobs=16)]: Done 210 tasks      | elapsed:   21.5s
[Parallel(n_jobs=16)]: Done 233 tasks      | elapsed:  


===== FINAL LOOCV PERFORMANCE =====
AUC      : 0.8230
Precision: 0.2460
Recall   : 0.3010
F1 Score : 0.2707
Accuracy : 0.9657


[Parallel(n_jobs=16)]: Done 4866 out of 4866 | elapsed:  7.6min finished


In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from sklearn.model_selection import LeaveOneOut
from joblib import Parallel, delayed
import xgboost as xgb
import multiprocessing
import warnings
warnings.filterwarnings("ignore")

# ======================================================
# Load data
# ======================================================
file_path = "C:/Users/aiden/School/6380/in-silico-drug-discovery/data/input_data.csv"
df = pd.read_csv(file_path)

labeled = df[df["senolytic"].isin([0, 1])].copy()
X = labeled.drop(columns=["senolytic", "ID", "SMILES"])
y = labeled["senolytic"].values

# ======================================================
# Scale
# ======================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ======================================================
# Feature selection (top 160)
# ======================================================
rf = RandomForestClassifier(n_estimators=500, n_jobs=-1, class_weight="balanced")
rf.fit(X_scaled, y)

importances = rf.feature_importances_
feat_idx_sorted = np.argsort(importances)[::-1]

top_k = 160
X_selected = X_scaled[:, feat_idx_sorted[:top_k]]

# ======================================================
# LOOCV setup
# ======================================================
loo = LeaveOneOut()
n_jobs = multiprocessing.cpu_count()

# ======================================================
# LOOCV evaluation
# ======================================================
def run_fold(train_idx, test_idx):
    X_train, X_test = X_selected[train_idx], X_selected[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    sample_weights = np.where(y_train == 1, 27, 1)

    model = xgb.XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        objective="binary:logistic",
        max_depth=3,
        n_estimators=200,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0,
        n_jobs=1
    )

    model.fit(X_train, y_train, sample_weight=sample_weights)
    
    y_prob = model.predict_proba(X_test)[0, 1]

    return y_test[0], y_prob

print("Running LOOCV evaluation...")

results = Parallel(n_jobs=n_jobs, verbose=10)(
    delayed(run_fold)(train_idx, test_idx)
    for train_idx, test_idx in loo.split(X_selected)
)

# ======================================================
# Collect results
# ======================================================
y_true, y_prob = zip(*results)

y_true = np.array(y_true)
y_prob = np.array(y_prob)

# ======================================================
# AUC (threshold independent)
# ======================================================
auc = roc_auc_score(y_true, y_prob)

# ======================================================
# Threshold tuning (optimize F1)
# ======================================================
thresholds = np.linspace(0.01, 0.99, 99)

best_f1 = 0
best_thresh = 0
best_metrics = None

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)

    p = precision_score(y_true, y_pred_t, zero_division=0)
    r = recall_score(y_true, y_pred_t, zero_division=0)
    f = f1_score(y_true, y_pred_t, zero_division=0)
    acc = accuracy_score(y_true, y_pred_t)

    if f > best_f1:
        best_f1 = f
        best_thresh = t
        best_metrics = (p, r, f, acc)

precision_best, recall_best, f1_best, accuracy_best = best_metrics

# ======================================================
# Print results
# ======================================================
print("\n===== FINAL LOOCV PERFORMANCE (OPTIMIZED THRESHOLD) =====")
print(f"Best Threshold: {best_thresh:.2f}")
print(f"AUC           : {auc:.4f}")
print(f"Precision     : {precision_best:.4f}")
print(f"Recall        : {recall_best:.4f}")
print(f"F1 Score      : {f1_best:.4f}")
print(f"Accuracy      : {accuracy_best:.4f}")

Running LOOCV evaluation...


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   9 tasks      | elapsed:    7.4s
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    8.5s
[Parallel(n_jobs=16)]: Done  29 tasks      | elapsed:    9.1s
[Parallel(n_jobs=16)]: Done  40 tasks      | elapsed:    9.9s
[Parallel(n_jobs=16)]: Done  53 tasks      | elapsed:   11.0s
[Parallel(n_jobs=16)]: Done  66 tasks      | elapsed:   12.0s
[Parallel(n_jobs=16)]: Done  81 tasks      | elapsed:   13.1s
[Parallel(n_jobs=16)]: Done  96 tasks      | elapsed:   14.2s
[Parallel(n_jobs=16)]: Done 113 tasks      | elapsed:   15.5s
[Parallel(n_jobs=16)]: Done 130 tasks      | elapsed:   16.7s
[Parallel(n_jobs=16)]: Done 149 tasks      | elapsed:   18.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:   19.3s
[Parallel(n_jobs=16)]: Done 189 tasks      | elapsed:   21.3s
[Parallel(n_jobs=16)]: Done 210 tasks      | elapsed:   22.7s
[Parallel(n_jobs=16)]: Done 233 tasks      | elapsed:  


===== FINAL LOOCV PERFORMANCE (OPTIMIZED THRESHOLD) =====
Best Threshold: 0.46
AUC           : 0.8324
Precision     : 0.2615
Recall        : 0.3301
F1 Score      : 0.2918
Accuracy      : 0.9661
